# 🛒 FMCG Sales Forecasting Project
**Goal:** Predict future retail/FMCG sales based on historical data while accounting for:
- 📈 **Trend Components** (long-term growth/decline)
- 🗓️ **Seasonal Components** (weekly shopping patterns)
- 🏷️ **Promotional Components** (`promotion_flag` uplift)
- 🤖 **Models Benchmarked:** SARIMAX (ARIMA), Facebook Prophet, and LSTM Neural Network
- 📊 **Visualizations:** Component decomposition and Forecast vs. Actual overlay

In [ ]:
# Step 0: Install dependencies in Google Colab
!pip install -q prophet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Set seeds
np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.figsize'] = (14, 5)

## 1. Data Ingestion & Daily Time-Series Aggregation
Because retail FMCG data contains multiple records per day (by SKU, region, channel), we aggregate to a continuous daily time series.

In [ ]:
# Load dataset (replace with your uploaded filename)
df = pd.read_csv('fmcg_sales.csv')
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df['revenue'] = df['units_sold'] * df['price_unit']

# Daily aggregation
daily_df = df.groupby('date').agg({
    'units_sold': 'sum',
    'revenue': 'sum',
    'promotion_flag': lambda x: 1 if (x == 1).any() else 0,
    'price_unit': 'mean'
}).reset_index()

# Set daily frequency
daily_df = daily_df.sort_values('date').set_index('date').asfreq('D')
daily_df['units_sold'] = daily_df['units_sold'].fillna(0)
daily_df['revenue'] = daily_df['revenue'].fillna(0)
daily_df['promotion_flag'] = daily_df['promotion_flag'].fillna(0)

print(f"Span: {daily_df.index.min().date()} to {daily_df.index.max().date()} ({len(daily_df)} days)")
daily_df.head()

## 2. Component Analysis (Trend, Seasonality & Promotional Lift)

In [ ]:
# 1. Calculate promotional lift
promo_stats = daily_df.groupby('promotion_flag')['units_sold'].mean()
lift = ((promo_stats[1] - promo_stats[0]) / promo_stats[0]) * 100
print(f"Avg Sales without Promo: {promo_stats[0]:.1f} units")
print(f"Avg Sales with Promo:    {promo_stats[1]:.1f} units")
print(f"Promotional Lift:        +{lift:.2f}%")

# 2. STL Decomposition
decomp = seasonal_decompose(daily_df['units_sold'], model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(daily_df['units_sold'], color='#1f77b4', label='Observed')
axes[0].set_title('Observed Sales', fontweight='bold')
axes[1].plot(decomp.trend, color='#ff7f0e', label='Trend')
axes[1].set_title('Trend Component', fontweight='bold')
axes[2].plot(decomp.seasonal, color='#2ca02c', label='Weekly Seasonality')
axes[2].set_title('Weekly Seasonal Pattern', fontweight='bold')
axes[3].scatter(daily_df.index, decomp.resid, color='#d62728', s=10, alpha=0.5)
axes[3].set_title('Residuals / Noise', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Train / Test Split
Time-series evaluation requires splitting chronologically (no random shuffle).

In [ ]:
test_days = 30
train_df = daily_df.iloc[:-test_days].copy()
test_df = daily_df.iloc[-test_days:].copy()

print(f"Training set: {train_df.index.min().date()} to {train_df.index.max().date()}")
print(f"Testing set:  {test_df.index.min().date()} to {test_df.index.max().date()}")

## 4. Model 1: SARIMAX (ARIMA with Exogenous Promotion)

In [ ]:
sarimax = SARIMAX(
    train_df['units_sold'],
    exog=train_df[['promotion_flag']],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

test_df['SARIMAX_Forecast'] = sarimax.predict(
    start=test_df.index[0],
    end=test_df.index[-1],
    exog=test_df[['promotion_flag']]
)
print("SARIMAX model completed.")

## 5. Model 2: Facebook Prophet (Multiplicative Mode with Promo Regressor)

In [ ]:
p_train = train_df.reset_index()[['date', 'units_sold', 'promotion_flag']].rename(
    columns={'date': 'ds', 'units_sold': 'y'}
)
m = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)
m.add_regressor('promotion_flag', mode='multiplicative')
m.fit(p_train)

p_test = test_df.reset_index()[['date', 'promotion_flag']].rename(columns={'date': 'ds'})
prophet_fc = m.predict(p_test)
test_df['Prophet_Forecast'] = prophet_fc['yhat'].values
print("Prophet model completed.")

## 6. Model 3: LSTM Neural Network (PyTorch)

In [ ]:
# Scale target and promotion features
scaler_y = MinMaxScaler()
scaler_p = MinMaxScaler()

train_y = scaler_y.fit_transform(train_df[['units_sold']])
test_y = scaler_y.transform(test_df[['units_sold']])
train_p = scaler_p.fit_transform(train_df[['promotion_flag']])
test_p = scaler_p.transform(test_df[['promotion_flag']])

train_data = np.hstack([train_y, train_p])
full_data = np.vstack([train_data, np.hstack([test_y, test_p])])

seq_len = 14
X_train, y_train = [], []
for i in range(len(train_data) - seq_len):
    X_train.append(train_data[i:i+seq_len])
    y_train.append(train_data[i+seq_len, 0])

X_train_t = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train_t = torch.tensor(np.array(y_train), dtype=torch.float32).unsqueeze(1)

class SalesLSTM(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm_model = SalesLSTM()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.01)
criterion = nn.MSELoss()
loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)

for epoch in range(50):
    lstm_model.train()
    for bx, by in loader:
        optimizer.zero_grad()
        loss = criterion(lstm_model(bx), by)
        loss.backward()
        optimizer.step()

# Forecast across test horizon
lstm_model.eval()
test_input = full_data[len(train_data) - seq_len:]
lstm_preds = []
with torch.no_grad():
    for i in range(len(test_df)):
        inp = torch.tensor(test_input[i:i+seq_len], dtype=torch.float32).unsqueeze(0)
        lstm_preds.append(lstm_model(inp).item())

test_df['LSTM_Forecast'] = scaler_y.inverse_transform(np.array(lstm_preds).reshape(-1, 1)).flatten()
print("LSTM model completed.")

## 7. Visualizations & Model Evaluation (Forecast vs. Actual)

In [ ]:
# 1. Plot Forecast vs Actual Overlay
plt.figure(figsize=(15, 6))
plt.plot(test_df.index, test_df['units_sold'], label='Actual Sales', color='black', lw=2.5, marker='o')
plt.plot(test_df.index, test_df['SARIMAX_Forecast'], label='SARIMAX (ARIMA)', color='#1f77b4', linestyle='--')
plt.plot(test_df.index, test_df['Prophet_Forecast'], label='Facebook Prophet', color='#2ca02c', linestyle='-.')
plt.plot(test_df.index, test_df['LSTM_Forecast'], label='LSTM Neural Net', color='#d62728', linestyle=':')

# Highlight promotional days
promo_dates = test_df[test_df['promotion_flag'] == 1].index
for pd_date in promo_dates:
    plt.axvline(pd_date, color='gold', alpha=0.35, linestyle='-', lw=4, label='Promotional Event' if pd_date == promo_dates[0] else "")

plt.title('Out-of-Sample Forecast vs. Actual Sales Across Models', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Daily Units Sold')
plt.legend()
plt.tight_layout()
plt.show()

# 2. Calculate Evaluation Metrics (RMSE, MAE, MAPE)
results = []
for col in ['SARIMAX_Forecast', 'Prophet_Forecast', 'LSTM_Forecast']:
    rmse = np.sqrt(mean_squared_error(test_df['units_sold'], test_df[col]))
    mae = mean_absolute_error(test_df['units_sold'], test_df[col])
    mape = mean_absolute_percentage_error(test_df['units_sold'], test_df[col]) * 100
    results.append({"Model": col.replace('_Forecast', ''), "RMSE": round(rmse, 2), "MAE": round(mae, 2), "MAPE (%)": round(mape, 2)})

metrics_table = pd.DataFrame(results)
print("\n--- Model Performance Comparison ---")
print(metrics_table.to_string(index=False))

# 3. Prophet Component Breakdown
fig_components = m.plot_components(prophet_fc)
plt.show()